In [ ]:
##Step 0: Install theses necesary packages
##Install google-api-python-client google-auth google-auth-oauthlib pandas beautifulsoup4 gspread
##install -q pillow

##Step 1: Import packages after installing

#Import for step 3
import re, unicodedata
import pandas as pd
#Import for step 5
import re
from urllib.parse import urlparse, parse_qs
#Import for step 6
import re
from googleapiclient.http import MediaInMemoryUpload
#Import for step 7
import os
from bs4 import BeautifulSoup
#Import for step 8
import base64
import io
import requests
#Import for step 9
import unicodedata
#Import for step 10
from urllib.parse import urlparse
#Import for step 11
from PIL import Image, UnidentifiedImageError
#Import for step 14
from bs4 import BeautifulSoup
import re
#Import for step 17
from bs4 import BeautifulSoup, NavigableString
import html
#Import for step 18
from bs4 import BeautifulSoup, NavigableString
from urllib.parse import urlparse, parse_qs, unquote
import re
#Import dotenv
import dotenv
dotenv.load_dotenv()

True

In [ ]:
##Step 2: Get credentials and build the drive client

# Reload functions
import importlib, functions
importlib.reload(functions) 

# Build drive
from Agno_agent_SEO_posting.functions import build_drive
build_drive()
print("Drive & Sheets clients ready")

# Get url and main keyword input
from Agno_agent_SEO_posting.functions import get_input_url_keyword
print("\n--- Input Google Docs URLs and Keywords ---")
print("For each document, enter the Google Docs URL and main keyword.")
print("Type 'done' when finished, or press Ctrl+D (Mac/Linux) / Ctrl+Z (Windows) on empty line:\n")
get_input_url_keyword()

Drive & Sheets clients ready

--- Input Google Docs URLs and Keywords ---
For each document, enter the Google Docs URL and main keyword.
Type 'done' when finished, or press Ctrl+D (Mac/Linux) / Ctrl+Z (Windows) on empty line:


--- Document 1 ---

Total documents collected: 0


In [ ]:
## Step 5: Export docs to html
# Export Google Docs → HTML (in-memory, no saving)

# Reload functions module for updates
import importlib, Agno_agent_SEO_posting.functions as functions
importlib.reload(functions) 

# Export html files from docs id
from Agno_agent_SEO_posting.functions import extract_file_id
from Agno_agent_SEO_posting.functions import export_doc_html_bytes
from Agno_agent_SEO_posting.functions import export_docs_to_html

# Export docs to html
export_docs_to_html()

RuntimeError: `docs_url_to_run` is not defined. Build it from your sheet first.

In [ ]:
## Step 6: Upload exported HTML to Google Drive

#Check if drive is available
try:
    drive  # Drive client from your auth step
    exported_html_docs  # list from Step 5
except NameError:
    raise RuntimeError("Missing `drive` or `exported_html_docs`. Run Step 5 first.")

# Load drive_folder_id
dotenv.load_dotenv()
DRIVE_FOLDER_ID = os.getenv("DRIVE_FOLDER_ID")

## Reload functions module for updates
import importlib, Agno_agent_SEO_posting.functions as functions
importlib.reload(functions) 
from Agno_agent_SEO_posting.functions import safe_filename

# Create list for storing uploaded html files
uploaded_html_files = []  # will hold dicts: {file_id, name, webViewLink, source_url, source_file_id}

for item in exported_html_docs:
    html_text = item["html"]
    base = safe_filename(item["name"])
    filename = f"{base}.html"

    media = MediaInMemoryUpload(
        html_text.encode("utf-8"),
        mimetype="text/html",
        resumable=False,
    )

    metadata = {
        "name": filename,
        "parents": [DRIVE_FOLDER_ID],
        "mimeType": "text/html",
    }

    try:
        file = (
            drive.files()
            .create(body=metadata, media_body=media, fields="id,name,webViewLink")
            # If uploading to a Shared Drive, uncomment the next line:
            # .create(body=metadata, media_body=media, fields="id,name,webViewLink", supportsAllDrives=True)
            .execute()
        )

        uploaded_html_files.append({
            "file_id": file["id"],
            "name": file["name"],
            "webViewLink": file.get("webViewLink"),
            "source_url": item["url"],
            "source_file_id": item["file_id"],
        })

        print(f"Uploaded: {file['name']}  →  {file['webViewLink']}")
    except Exception as e:
        print(f"Failed to upload {filename}: {e}")

print(f"\nDone. Uploaded {len(uploaded_html_files)} HTML file(s) to Drive folder {DRIVE_FOLDER_ID}.")

Uploaded: Bviet.-Xe-máy-điện-là-gì.html  →  https://drive.google.com/file/d/1ScVil94nf7YHy8mYOe3xTLbC9pfn0U_0/view?usp=drivesdk

Done. Uploaded 1 HTML file(s) to Drive folder 1NjOhY5tEOFUrz49YeQiA_fwGzKg_4CJX.


In [ ]:
## Step 7: Process each local HTML file and extract <img> tags
from bs4 import BeautifulSoup

# Expect: exported_html_docs = [
#   {"url": ..., "file_id": ..., "name": ..., "html": "<!doctype html>...", "main_keyword": ...},
#   ...
# ]

# Check if the html_docs is available
try:
    exported_html_docs  # list built in Step 5
except NameError:
    raise RuntimeError("`exported_html_docs` not found. Run the export step first.")

# Reload functions module for updates
import importlib, Agno_agent_SEO_posting.functions as functions
importlib.reload(functions) 

# Import functions
from Agno_agent_SEO_posting.functions import extract_images_from_html_text

images_each_doc = []   # [{name, url, file_id, main_keyword, images:[{...}, ...]}, ...]
total_imgs = 0

for item in exported_html_docs:
    html_text = item.get("html", "") or ""
    imgs = extract_images_from_html_text(html_text)
    images_each_doc.append({
        "name": item.get("name"),
        "url": item.get("url"),
        "file_id": item.get("file_id"),
        "main_keyword": item.get("main_keyword"),
        "images": imgs,
    })
    total_imgs += len(imgs)

print(f"Extracted {total_imgs} <img> tag(s) across {len(images_each_doc)} HTML document(s).")
images_each_doc

Extracted 9 <img> tag(s) across 1 HTML document(s).


[{'name': 'Bviet. Xe máy điện là gì',
  'url': 'https://docs.google.com/document/d/18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U/edit?tab=t.0',
  'file_id': '18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U',
  'main_keyword': 'test bài đăng',
  'images': [{'alt': 'Xe máy điện sử dụng năng lượng thân thiện môi trường với tốc độ tối đa không quá 50km/h',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXdI2ScHkwYYXnacI0FzL4RatDbr1gp81IbYFXUkXcSjmCaBNnegicmwOu_07qenIv10M8ED4oagNAc9GKZ0Vt9N4nawVz1vFN44JKGFQipIb7bzAqQLSEIVZW9ilNw0fXzPVDP1yg?key=jy2_nGf6-COJ3m01OoJ7dWra',
    'style': 'width: 601.70px; height: 397.33px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': ''},
   {'alt': '',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXfi-UX6m7avn8kSTf3yhOMECN_gu32eQSa6-i_eGSMMF5knIri3slSMRHrgiM40Y3YLSzNN-Pq8FtnngjY_0GglPD12Gvdb8FhZ2BI0avGRd4JwEUq7UAmo0UUFJ0sZlRBDasiBmw?k

In [ ]:
## Step 8: Generate main_keyword_slug and alt_slug

# Check if images_each_doc is available
try:
    images_each_doc  # from Step 7
except NameError:
    raise RuntimeError("`images_each_doc` not found. Run the image extraction step first.")

# Reload functions
importlib.reload(functions) 

# Import functions
from Agno_agent_SEO_posting.functions import strip_diacritics
from Agno_agent_SEO_posting.functions import slugify
from Agno_agent_SEO_posting.functions import first_n_words

# Create list of images in each doc with src, alt, slugs attribute
images_with_slugs = []  # [{doc_name, url, file_id, main_keyword, main_keyword_slug, images:[{src, alt, main_keyword_slug, alt_slug, ...}]}]
for doc in images_each_doc:
    # Slugify the main keyword
    main_keyword = doc.get("main_keyword", "") or ""
    base_keyword_slug = slugify(main_keyword)
    
    doc_entry = {
        "doc_name": doc.get("name"),
        "url": doc.get("url"),
        "file_id": doc.get("file_id"),
        "main_keyword": main_keyword,
        "main_keyword_slug": base_keyword_slug,  # Store the general slug at doc level
        "images": []
    }
    
    for i, attrs in enumerate(doc.get("images", []), start=1):
        alt = attrs.get("alt", "") or ""
        
        # Create main_keyword_slug with number for each image
        main_keyword_slug = f"{base_keyword_slug}-{i}" if base_keyword_slug else f"image-{i}"
        
        # Create alt_slug from first 5 words of alt text
        alt_slug = slugify(first_n_words(alt, 5))
        
        # copy existing attrs and add slugs
        enriched = dict(attrs)
        enriched["main_keyword_slug"] = main_keyword_slug
        enriched["alt_slug"] = alt_slug
        doc_entry["images"].append(enriched)
    
    images_with_slugs.append(doc_entry)

# Quick peek
total_imgs = sum(len(d["images"]) for d in images_with_slugs)
print(f"Added slugs for {total_imgs} image(s) across {len(images_with_slugs)} document(s).")

# Show examples
for d in images_with_slugs[:1]:  # Show first doc only
    print(f"\nDocument: {d['doc_name']}")
    print(f"Main keyword: '{d['main_keyword']}'")
    print(f"Main keyword slug: '{d['main_keyword_slug']}'")
    for img in d['images'][:3]:  # Show first 3 images
        print(f"  - Image slug: '{img['main_keyword_slug']}', alt_slug: '{img['alt_slug']}'")

images_with_slugs

tet-phang-sun
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=467602322265-sbnjhkbjod1hcegcgl4prdhj0f8papoe.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A60371%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets.readonly&state=skDAkJPLOIXZRhkfgUNTZuLXsfaElG&prompt=consent&access_type=offline
Added slugs for 9 image(s) across 1 document(s).

Document: Bviet. Xe máy điện là gì
Main keyword: 'test bài đăng'
Main keyword slug: 'test-bai-dang'
  - Image slug: 'test-bai-dang-1', alt_slug: 'xe-may-dien-su-dung'
  - Image slug: 'test-bai-dang-2', alt_slug: ''
  - Image slug: 'test-bai-dang-3', alt_slug: 'xe-may-dien-so-huu'


[{'doc_name': 'Bviet. Xe máy điện là gì',
  'url': 'https://docs.google.com/document/d/18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U/edit?tab=t.0',
  'file_id': '18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U',
  'main_keyword': 'test bài đăng',
  'main_keyword_slug': 'test-bai-dang',
  'images': [{'alt': 'Xe máy điện sử dụng năng lượng thân thiện môi trường với tốc độ tối đa không quá 50km/h',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXdI2ScHkwYYXnacI0FzL4RatDbr1gp81IbYFXUkXcSjmCaBNnegicmwOu_07qenIv10M8ED4oagNAc9GKZ0Vt9N4nawVz1vFN44JKGFQipIb7bzAqQLSEIVZW9ilNw0fXzPVDP1yg?key=jy2_nGf6-COJ3m01OoJ7dWra',
    'style': 'width: 601.70px; height: 397.33px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'main_keyword_slug': 'test-bai-dang-1',
    'alt_slug': 'xe-may-dien-su-dung'},
   {'alt': '',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXfi-UX6m7avn

In [ ]:
## Step 09: Download and resize images in one step (memory-efficient)

#Check if images_with_slugs is empty
try:
    images_with_slugs  # [{doc_name, url, file_id, main_keyword, images:[{src, alt, main_keyword_slug, alt_slug, ...}]}]
except NameError:
    raise RuntimeError("`images_with_slugs` not found. Run the slug-building step first.")

# Define target width and height for resizing
target_image_width = 800  # pixels
target_image_height = 600 # pixels

# Only save resized images
RESIZED_DIR = "images_resized"
os.makedirs(RESIZED_DIR, exist_ok=True)

resized_image_paths = {}  # {src_url: resized_local_path}
resize_errors = []

importlib.reload(functions)
from Agno_agent_SEO_posting.functions import safe_name
from Agno_agent_SEO_posting.functions import ensure_unique_path
from Agno_agent_SEO_posting.functions import guess_ext_from_url
from Agno_agent_SEO_posting.functions import download_to_memory  # Download to BytesIO
from Agno_agent_SEO_posting.functions import resize_fit_memory   # Resize from memory

total_attempts = 0
print("\n--- Downloading and resizing images (memory-efficient) ---")

for doc in images_with_slugs:
    doc_name = safe_name(doc.get("doc_name") or doc.get("file_id") or "document")
    
    imgs = doc.get("images", []) or []
    if not imgs:
        continue

    print(f"\n{doc_name}: {len(imgs)} image(s)")

    for attrs in imgs:
        src = attrs.get("src")
        if not src:
            continue

        # Use main_keyword_slug as the base filename
        base = attrs.get("main_keyword_slug", "image")
        ext = guess_ext_from_url(src)
        filename = f"{base}{ext}"
        dst = os.path.join(RESIZED_DIR, filename)
        dst = ensure_unique_path(RESIZED_DIR, filename)

        total_attempts += 1
        try:
            print(f"  ↓ {src}")
            # Download to memory, resize, and save
            img_bytes = download_to_memory(src, timeout=60)
            resize_fit_memory(img_bytes, dst, target_image_width, target_image_height)
            resized_image_paths[src] = dst
            print(f"    ✓ Resized → {dst}")
        except Exception as e:
            resize_errors.append({"src": src, "filename": filename, "doc_name": doc_name, "error": str(e)})
            print(f"    ✗ Failed: {e}")

print("\n--- Summary ---")
print(f"Total attempted: {total_attempts}")
print(f"Successfully resized: {len(resized_image_paths)}")
print(f"Failed: {len(resize_errors)}")



--- Downloading and resizing images (memory-efficient) ---

Bviet. Xe máy điện là gì: 9 image(s)
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXdI2ScHkwYYXnacI0FzL4RatDbr1gp81IbYFXUkXcSjmCaBNnegicmwOu_07qenIv10M8ED4oagNAc9GKZ0Vt9N4nawVz1vFN44JKGFQipIb7bzAqQLSEIVZW9ilNw0fXzPVDP1yg?key=jy2_nGf6-COJ3m01OoJ7dWra
    ✓ Resized → images_resized/test-bai-dang-1.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXfi-UX6m7avn8kSTf3yhOMECN_gu32eQSa6-i_eGSMMF5knIri3slSMRHrgiM40Y3YLSzNN-Pq8FtnngjY_0GglPD12Gvdb8FhZ2BI0avGRd4JwEUq7UAmo0UUFJ0sZlRBDasiBmw?key=jy2_nGf6-COJ3m01OoJ7dWra
    ✓ Resized → images_resized/test-bai-dang-2.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXcLsN7zedMpgiJXhDtDM1N-zP09SoWwBrkLhaie1fHqQj7UYEiXV8KdVB5zY3DuMEL1IagMNd_M3O_VXChN2I__8flofqKRR08qKbRo01wVdJJ78UVn13PLTiZ_DysfOijRzgJQ5A?key=jy2_nGf6-COJ3m01OoJ7dWra
    ✓ Resized → images_resized/test-bai-dang-3.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXfS9KUhqYNnQqP-54TT2Kc_0NNe5BBH3oAkN_KXn

In [63]:
## Step 11: Setup Wordpress API

# Reload .env file to get fresh values
import dotenv
dotenv.load_dotenv(override=True)  # Force reload with override=True

# Set up Wordpress API
WP_BASE_URL = "https://ngoncareer.com"          # No trailing slash
WP_USERNAME  = "admin_career"
WP_APP_PASS = os.getenv("WP_APP_PASS")
wp_pass_clean = (WP_APP_PASS or "").replace(" ", "")  # WP_APP_PASS defined in environment

# Check if API is missing some variables
if not WP_BASE_URL or not WP_USERNAME or not wp_pass_clean:
    raise ValueError("Missing WP_BASE_URL / WP_USERNAME / WP_APP_PASS")

# Build the REST base and commonly used endpoints
WP_API_BASE = f"{WP_BASE_URL}/wp-json/wp/v2"
WP_MEDIA_EP = f"{WP_API_BASE}/media"

# Build Basic Auth header
token = base64.b64encode(f"{WP_USERNAME}:{wp_pass_clean}".encode("utf-8")).decode("utf-8")
auth_header = {"Authorization": f"Basic {token}"}

# Debug: Print password info
print("Debug - Password info:")
print(f"  WP_APP_PASS: '{WP_APP_PASS}'")
print(f"  Password length: {len(WP_APP_PASS)}")
print(f"  Clean password: '{wp_pass_clean}'")

# Final check
print("\nWordPress REST API auth header ready.")
print("Base:", WP_API_BASE)

Debug - Password info:
  WP_APP_PASS: 'efkV mie6 u3P8 C3mC NyM8 Ickp'
  Password length: 29
  Clean password: 'efkVmie6u3P8C3mCNyM8Ickp'

WordPress REST API auth header ready.
Base: https://ngoncareer.com/wp-json/wp/v2


In [ ]:
print(f"Password from .env: '{WP_APP_PASS}'")
print(f"Password length: {len(WP_APP_PASS)}")
print(f"Clean password: '{wp_pass_clean}'")
print(f"Auth header: {auth_header}")

In [ ]:
#Step 12: Upload images to wordpress

#Check if the API endpoint, API authentication, resized images are all available
try:
    WP_MEDIA_EP, auth_header, resized_image_paths
except NameError:
    raise RuntimeError("Missing WP_MEDIA_EP/auth_header or resized_image_paths.")

#Import necessary functions
importlib.reload(functions)
from Agno_agent_SEO_posting.functions import guess_mime        #Get file extension 
from Agno_agent_SEO_posting.functions import upload_resized    #Uploading resized images

# Results:
# - wp_uploaded_map: {original_src_url: {"id": <media_id>, "wp_url": <public_url>, "name": <title>}}
# - wp_upload_errors: [{source_url, local_path, error}]
wp_uploaded_map = {}
wp_upload_errors = []

print("\n— Uploading RESIZED images to WordPress —")
for orig_src, local_path in resized_image_paths.items():
    try:
        media = upload_resized(local_path)
        wp_uploaded_map[orig_src] = {
            "id": media.get("id"),
            "wp_url": media.get("source_url"),
            "name": (media.get("title") or {}).get("rendered") or media.get("slug") or os.path.basename(local_path),
            "mime_type": media.get("mime_type"),
            "local_path": local_path,
        }
        print(f"✓ {os.path.basename(local_path)} → {wp_uploaded_map[orig_src]['wp_url']}")
    except Exception as e:
        wp_upload_errors.append({"source_url": orig_src, "local_path": local_path, "error": str(e)})
        print(f"✗ Failed: {local_path} | {e}")

print(f"\nUpload summary: uploaded={len(wp_uploaded_map)} | failed={len(wp_upload_errors)}")
# Now you can use `wp_uploaded_map[original_src]["wp_url"]` to update images_with_slugs later.



— Uploading RESIZED images to WordPress —
✓ test-bai-dang-1.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-1.jpg
✓ test-bai-dang-2.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-2.jpg
✓ test-bai-dang-3.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-3.jpg
✓ test-bai-dang-4.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-4.jpg
✓ test-bai-dang-5.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-5.jpg
✓ test-bai-dang-6.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-6.jpg
✓ test-bai-dang-7.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-7.jpg
✓ test-bai-dang-8.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-8.jpg
✓ test-bai-dang-9.jpg → https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-9.jpg

Upload summary: uploaded=9 | failed=0


In [65]:
## Step 13: Update images_with_slugs variable
## Step: Clean + update images_with_slugs
# - Ensure no pre-existing `width` / `height` on each image, then add fresh values
# - For `new_src`: if it exists, remove it first; then set it to the new WordPress URL

import os

# Pillow for reading actual resized dimensions
try:
    from PIL import Image
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow"])
    from PIL import Image

# Requires:
# - images_with_slugs: [{doc_name, url, file_id, images:[{src, ...}]}]
# - resized_image_paths: {original_src_url: "/path/to/resized.ext"}
# - wp_uploaded_map: {original_src_url: {"id":..., "wp_url":..., "local_path":...}}

try:
    images_with_slugs, resized_image_paths, wp_uploaded_map
except NameError:
    raise RuntimeError("Missing images_with_slugs, resized_image_paths, or wp_uploaded_map.")

updated = 0
skipped = 0
errors = []

for doc in images_with_slugs:
    for img in doc.get("images", []):
        orig_src = img.get("src")
        if not orig_src:
            skipped += 1
            continue

        up = wp_uploaded_map.get(orig_src)
        local_resized = resized_image_paths.get(orig_src)

        if not up or not local_resized or not os.path.isfile(local_resized):
            # No upload info or no resized file → cannot update
            skipped += 1
            continue

        # --- 1) remove any existing width/height first
        img.pop("width", None)
        img.pop("height", None)

        # --- 2) remove existing new_src (if present), then add new one
        img.pop("new_src", None)
        img["new_src"] = up.get("wp_url")

        # --- 3) compute actual resized dimensions and add width/height
        try:
            with Image.open(local_resized) as im:
                w, h = im.size
            img["width"] = int(w)
            img["height"] = int(h)
            # optional helpful fields:
            img["uploaded_media_id"] = up.get("id")
            img["resized_local_path"] = local_resized
            updated += 1
        except Exception as e:
            errors.append({"src": orig_src, "path": local_resized, "error": str(e)})

print(f"Cleaned & updated {updated} image(s). Skipped: {skipped}. Errors: {len(errors)}")
# Optional quick peek:
# [ (img.get('src'), img.get('new_src'), img.get('width'), img.get('height'))
#   for d in images_with_slugs for img in d['images'][:3] ]
images_with_slugs

Cleaned & updated 9 image(s). Skipped: 0. Errors: 0


[{'doc_name': 'Bviet. Xe máy điện là gì',
  'url': 'https://docs.google.com/document/d/18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U/edit?tab=t.0',
  'file_id': '18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U',
  'main_keyword': 'test bài đăng',
  'main_keyword_slug': 'test-bai-dang',
  'images': [{'alt': 'Xe máy điện sử dụng năng lượng thân thiện môi trường với tốc độ tối đa không quá 50km/h',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXdI2ScHkwYYXnacI0FzL4RatDbr1gp81IbYFXUkXcSjmCaBNnegicmwOu_07qenIv10M8ED4oagNAc9GKZ0Vt9N4nawVz1vFN44JKGFQipIb7bzAqQLSEIVZW9ilNw0fXzPVDP1yg?key=jy2_nGf6-COJ3m01OoJ7dWra',
    'style': 'width: 601.70px; height: 397.33px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'main_keyword_slug': 'test-bai-dang-1',
    'alt_slug': 'xe-may-dien-su-dung',
    'new_src': 'https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-1.jpg',


In [73]:
## Step 14: Extract the main heading from each exported HTML and append to docs_url_to_run

try:
    exported_html_docs  # [{url, file_id, name, html, main_keyword}]
    docs_url_to_run  # [{url, main_keyword}]
except NameError:
    raise RuntimeError("`exported_html_docs` or `docs_url_to_run` not found. Run previous steps first.")

def clean_text(s: str) -> str:
    """Clean text: collapse whitespace and remove comments"""
    # Remove HTML comments
    s = re.sub(r'<!--.*?-->', '', s or '', flags=re.DOTALL)
    # Remove comment-like patterns: [comment], (comment), {comment}
    s = re.sub(r'\[.*?\]', '', s)
    s = re.sub(r'\(.*?\)', '', s)
    s = re.sub(r'\{.*?\}', '', s)
    # Collapse multiple spaces into one
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# Create a mapping of URL to h1
url_to_h1 = {}

for doc in exported_html_docs:
    html = doc.get("html", "") or ""
    soup = BeautifulSoup(html, "html.parser")

    # only the FIRST <h1>
    h1_tag = soup.find("h1")
    h1_text = clean_text(h1_tag.get_text()) if h1_tag and h1_tag.get_text(strip=True) else None

    # Store in mapping
    url = doc.get("url")
    if url:
        url_to_h1[url] = h1_text

    if h1_text is None:
        print(f"(no <h1>) {doc.get('name') or doc.get('file_id')}")

# Append h1 to docs_url_to_run
for doc in docs_url_to_run:
    url = doc.get("url")
    doc["h1"] = url_to_h1.get(url)

print(f"Extracted first <h1> for {len(url_to_h1)} document(s) and appended to docs_url_to_run.")

# Show the updated structure
for doc in docs_url_to_run:
    print(f"  URL: {doc['url']}")
    print(f"    Main keyword: {doc['main_keyword']}")
    print(f"    H1: {doc['h1']}")
    
docs_url_to_run

Extracted first <h1> for 1 document(s) and appended to docs_url_to_run.
  URL: https://docs.google.com/document/d/18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U/edit?tab=t.0
    Main keyword: test bài đăng
    H1: Xe máy điện là gì? Cách phân biệt xe máy điện và xe đạp điện


[{'url': 'https://docs.google.com/document/d/18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U/edit?tab=t.0',
  'main_keyword': 'test bài đăng',
  'h1': 'Xe máy điện là gì? Cách phân biệt xe máy điện và xe đạp điện'}]

In [74]:
##Step 15: Create empty post in wordpress
# Requires:
# - WP_API_BASE and auth_header from your Step 11
# - docs_url_to_run (with h1)
# - images_with_slugs (with main_keyword_slug)

try:
    WP_API_BASE, auth_header, docs_url_to_run, images_with_slugs
except NameError:
    raise RuntimeError("Missing WP_API_BASE/auth_header, docs_url_to_run, or images_with_slugs.")

posts_endpoint = f"{WP_API_BASE}/posts"

# Get title (h1) from docs_url_to_run and main_keyword_slug from images_with_slugs
title = (docs_url_to_run[0].get("h1") or "").strip()
main_keyword_slug = images_with_slugs[0].get("main_keyword_slug", "")  # Use the slug from Step 8

post_data = {
    "title": title,
    "slug": main_keyword_slug,
    "status": "draft",
    "content": "",
}

print(f"Creating draft post at: {posts_endpoint}")
print(f"Title: {title}")
print(f"Slug: {main_keyword_slug}")

resp = requests.post(posts_endpoint, headers=auth_header, json=post_data)

if resp.status_code in (200, 201):
    data = resp.json()
    wordpress_post_id = data.get("id")
    wordpress_post_link = data.get("link")
    print(f"✓ Draft created. ID: {wordpress_post_id} | Link: {wordpress_post_link}")
else:
    raise RuntimeError(f"Post create failed {resp.status_code}: {resp.text[:500]}")

Creating draft post at: https://ngoncareer.com/wp-json/wp/v2/posts
Title: Xe máy điện là gì? Cách phân biệt xe máy điện và xe đạp điện
Slug: test-bai-dang
✓ Draft created. ID: 51902 | Link: https://ngoncareer.com/?p=51902


In [68]:
##Step 16: Update images metadata
# Requires:
#   - WP_API_BASE or WP_MEDIA_EP, and auth_header (from Step 11)
#   - wordpress_post_id (created post ID)
#   - wp_uploaded_map: {original_src_url: {"id": <media_id>, "wp_url": <media_url>, ...}}
# Optional:
#   - images_with_slugs to supply alt text per original src

# ---- inputs check ----
try:
    wordpress_post_id
    auth_header
    WP_API_BASE
except NameError:
    raise RuntimeError("Missing wordpress_post_id, auth_header, or WP_API_BASE (from Step 11).")

WP_MEDIA_EP = f"{WP_API_BASE}/media"

try:
    wp_uploaded_map
except NameError:
    raise RuntimeError("Missing `wp_uploaded_map` (built after uploads).")

# Build src_url -> alt text map from images_with_slugs (optional)
src_to_alt = {}
if "images_with_slugs" in globals() and isinstance(images_with_slugs, list):
    for doc in images_with_slugs:
        for img in doc.get("images", []):
            src = img.get("src")
            if not src:
                continue
            alt = (img.get("alt") or "").strip()
            # first win
            src_to_alt.setdefault(src, alt)

updated, failed = 0, []
print("\n— Updating media metadata and associating with the post —")

for original_src, meta in wp_uploaded_map.items():
    media_id = meta.get("id")
    if not media_id:
        failed.append({"src": original_src, "error": "Missing media_id"})
        continue

    alt = src_to_alt.get(original_src, "").strip()

    payload = {
        "alt_text": alt,                 # alt on image
        "caption": alt,                  # caption text
        "description": alt,              # description text
        "post": int(wordpress_post_id),  # attach to the created post
    }

    try:
        r = requests.post(
            f"{WP_MEDIA_EP}/{media_id}",
            headers={**auth_header, "Content-Type": "application/json"},
            json=payload,
            timeout=60,
        )
        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
        updated += 1
        print(f"✓ Media {media_id} updated")
    except Exception as e:
        failed.append({"media_id": media_id, "src": original_src, "error": str(e)})
        print(f"✗ Media {media_id} failed: {e}")

print(f"\nSummary: updated={updated} | failed={len(failed)}")
# Optional peek:
# failed[:3]

# Append the WordPress post_id onto every image entry in images_with_slugs
## Append/refresh WordPress post_id on images_with_slugs (no duplicates)

try:
    wordpress_post_id
    images_with_slugs
except NameError:
    raise RuntimeError("Missing `wordpress_post_id` or `images_with_slugs`.")

pid = int(wordpress_post_id)
set_count = 0
already_count = 0
corrected_count = 0

for doc in images_with_slugs:
    for img in doc.get("images", []):
        if "post_id" in img:
            # If it's different (or wrong type), refresh it; otherwise leave as-is
            if img["post_id"] != pid:
                img["post_id"] = pid
                corrected_count += 1
            else:
                already_count += 1
        else:
            img["post_id"] = pid
            set_count += 1

print(
    f"post_id updates → set:{set_count}, corrected:{corrected_count}, already-correct:{already_count}"
)
images_with_slugs


— Updating media metadata and associating with the post —
✓ Media 51891 updated
✓ Media 51892 updated
✓ Media 51893 updated
✓ Media 51894 updated
✓ Media 51895 updated
✓ Media 51896 updated
✓ Media 51897 updated
✓ Media 51898 updated
✓ Media 51899 updated

Summary: updated=9 | failed=0
post_id updates → set:9, corrected:0, already-correct:0


[{'doc_name': 'Bviet. Xe máy điện là gì',
  'url': 'https://docs.google.com/document/d/18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U/edit?tab=t.0',
  'file_id': '18q5qtDLxV2SoZPTGEX16r2cGTq3m9xZTtA9Y_SLMZ4U',
  'main_keyword': 'test bài đăng',
  'main_keyword_slug': 'test-bai-dang',
  'images': [{'alt': 'Xe máy điện sử dụng năng lượng thân thiện môi trường với tốc độ tối đa không quá 50km/h',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXdI2ScHkwYYXnacI0FzL4RatDbr1gp81IbYFXUkXcSjmCaBNnegicmwOu_07qenIv10M8ED4oagNAc9GKZ0Vt9N4nawVz1vFN44JKGFQipIb7bzAqQLSEIVZW9ilNw0fXzPVDP1yg?key=jy2_nGf6-COJ3m01OoJ7dWra',
    'style': 'width: 601.70px; height: 397.33px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'main_keyword_slug': 'test-bai-dang-1',
    'alt_slug': 'xe-may-dien-su-dung',
    'new_src': 'https://ngoncareer.com/wp-content/uploads/2025/10/test-bai-dang-1.jpg',


In [ ]:
# Step 17 (VS Code): Replace <img> tags with WordPress [caption] shortcodes (src + alt + width + height)
# Strategy:
#   1) Try exact per-doc match by original `src` from images_with_slugs.
#   2) If no exact match (Docs regenerates URLs), fall back to order: 1st <img> ↔ 1st image data, etc.

from bs4 import BeautifulSoup, NavigableString
from html import unescape as html_unescape  # avoid shadowing the html module

# Requires:
# - exported_html_docs: [{file_id, name, html, ...}]
# - images_with_slugs : [{file_id or doc_name/name/url, images:[{src, new_src/new_url, alt, width/new_width, height/new_height, uploaded_media_id}, ...]}]

try:
    exported_html_docs, images_with_slugs
except NameError:
    raise RuntimeError("Missing `exported_html_docs` or `images_with_slugs`. Run previous steps first.")

DEFAULT_ALIGN = "aligncenter"
DEFAULT_SIZE_CLASS = "size-full"

# Group images_with_slugs by file_id (preferred) and by name (fallback)
by_file_id, by_name = {}, {}
for d in images_with_slugs:
    key_id = d.get("file_id")
    key_name = d.get("doc_name") or d.get("name") or d.get("url")
    images = d.get("images", []) or []
    if key_id:
        by_file_id.setdefault(key_id, []).extend(images)
    if key_name:
        by_name.setdefault(key_name, []).extend(images)

#Import functions
importlib.reload(functions)
from Agno_agent_SEO_posting.functions import build_caption_shortcode

processed_html_docs = []   # [{file_id, name, html_processed, images_replaced}]
total_replaced = 0

for doc in exported_html_docs:
    file_id = doc.get("file_id")
    name    = doc.get("name")
    html_in = doc.get("html") or ""

    soup = BeautifulSoup(html_in, "html.parser")
    img_tags = soup.find_all("img")

    # per-doc image data
    imgs_list = by_file_id.get(file_id)
    if imgs_list is None:
        imgs_list = by_name.get(name, [])
    per_doc_src_map = { img.get("src"): img for img in imgs_list if img.get("src") }

    n_html = len(img_tags)
    n_data = len(imgs_list)
    n_replace = min(n_html, n_data)  # fallback upper-bound when no exact match
    replaced = 0
    idx_fallback = 0

    for i, tag in enumerate(img_tags):
        orig_src = tag.get("src")
        matched_info = None

        # 1) exact per-doc src match
        if orig_src in per_doc_src_map:
            matched_info = per_doc_src_map[orig_src]

        # 2) fallback by position if still no match
        if matched_info is None and idx_fallback < n_data:
            matched_info = imgs_list[idx_fallback]
            idx_fallback += 1

        if not matched_info:
            continue
        if not (matched_info.get("new_src") or matched_info.get("new_url")):
            continue

        shortcode = build_caption_shortcode(matched_info)
        tag.replace_with(NavigableString(shortcode))
        replaced += 1

    processed_html_docs.append({
        "file_id": file_id,
        "name": name,
        "html_processed": html_unescape(str(soup)),  # ensure shortcodes not HTML-escaped
        "images_replaced": replaced,
    })
    total_replaced += replaced

print(f"Processed {len(processed_html_docs)} document(s). Total images replaced: {total_replaced}")

# Optional: handy dict for further steps (e.g., updating WP post content)
processed_html_contents = { (d["file_id"] or d["name"]): d["html_processed"] for d in processed_html_docs }


Processed 1 document(s). Total images replaced: 9


In [ ]:
# Step 18: Robust HTML transforms (VS Code friendly)
from bs4 import BeautifulSoup, NavigableString
from urllib.parse import urlparse, parse_qs, unquote
import re

#Import functions
importlib.reload(functions)
from Agno_agent_SEO_posting.functions import _parse_style
from Agno_agent_SEO_posting.functions import _style_to_str
from Agno_agent_SEO_posting.functions import transform_html_dom

In [71]:
# Step 19: Apply changes

#Check if the html content is available
try:
    processed_html_docs
except NameError:
    raise RuntimeError("Run the image-replacement step first to build `processed_html_docs`.")

changed = 0
for d in processed_html_docs:
    src = d.get("html_processed") or d.get("html") or ""
    out = transform_html_dom(src)
    if out != src:
        changed += 1
    d["html_processed"] = out

print(f"Transformed {changed}/{len(processed_html_docs)} document(s).")

Transform summary: {'cmnt_blocks_removed': 12, 'h1_removed': 1, 'p_normalized': 36, 'p_img_centered_wrapped': 9, 'style_tags_removed': 1, 'span_bold_to_strong': 28, 'span_unwrapped': 102, 'links_unwrapped': 19, 'font_props_removed': 76}
Transformed 1/1 document(s).


In [72]:
## Step 20: Update content of post in wordpress

# Requires:
# - WP_API_BASE, auth_header (from Step 11)
# - wordpress_post_id (from Step 15)
# - processed_html_docs (from the replace step)

try:
    WP_API_BASE, auth_header, wordpress_post_id, processed_html_docs
except NameError:
    raise RuntimeError("Missing WP_API_BASE/auth_header, wordpress_post_id, or processed_html_docs.")

if not processed_html_docs:
    raise RuntimeError("processed_html_docs is empty. Run the HTML processing step first.")

# Choose which processed doc to insert (here: the first one)
html_str = processed_html_docs[0]["html_processed"]

posts_endpoint = f"{WP_API_BASE}/posts/{int(wordpress_post_id)}"
payload = {
    "content": html_str,   # raw HTML (can include shortcodes)
    # "status": "draft",   # optional: keep as draft
    # "status": "publish"  # optional: publish immediately
}

print(f"Updating post #{wordpress_post_id} at: {posts_endpoint}")
resp = requests.post(
    posts_endpoint,
    headers={**auth_header, "Content-Type": "application/json"},
    json=payload,
    timeout=90,
)

if resp.status_code == 200:
    data = resp.json()
    print(f"✓ Post updated. View: {data.get('link')}")
else:
    raise RuntimeError(f"Update failed {resp.status_code}: {resp.text[:500]}")


Updating post #51900 at: https://ngoncareer.com/wp-json/wp/v2/posts/51900
✓ Post updated. View: https://ngoncareer.com/?p=51900
